# unit03 レッスン: セレクタと表・リストの抽出

**このレッスンで作れるようになるもの**: 「同じ構造が繰り返されたHTML」(商品一覧・統計表・おすすめリスト)から、CSSセレクタで一括抽出し、表を `list[dict]`(1行=1辞書)に整形し、欠損セルがあっても落ちない防御的なコードを書く — スクレイピングの「量をさばく」部分の中核。

unit02 で `find` / `find_all` を1件ずつ辿る方法を学びました。今回はその**上位互換の絞り込み言語=CSSセレクタ**を手に入れ、「このクラスの要素**すべて**」を1行で拾えるようになります。さらに整形の定番ゴール `list[dict]` と、実データの汚れ(欠損・クラス揺れ)への耐性を身につけます。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
from bs4 import BeautifulSoup

def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

# このレッスンで題材にするHTML。演習の data/products.html を縮小したものです。
# 演習では BeautifulSoup(path.read_text(...), "html.parser") でファイルから読みますが、
# ここでは cwd に依存しないよう「取得済みの文字列」として手元に持っておきます(中身の構造は同じ)。
PRODUCTS_HTML = """<html><body>
<h1>コーヒー豆の販売一覧</h1>
<div class="product-grid">
    <div class="product-card" data-id="1">
        <h2 class="product-name">エチオピア モカ</h2>
        <p class="product-price">1200円</p>
        <p class="product-stock">在庫あり</p>
    </div>
    <div class="product-card" data-id="2">
        <h2 class="product-name">ブラジル サントス</h2>
        <p class="product-price">980円</p>
        <p class="product-stock">在庫あり</p>
    </div>
    <div class="product-card" data-id="3">
        <h2 class="product-name">グアテマラ アンティグア</h2>
        <p class="product-price">1450円</p>
        <p class="product-stock">在庫切れ</p>
    </div>
</div>
</body></html>"""

# 演習の data/table_stats.html を縮小したもの(月間売上の統計表)
TABLE_HTML = """<html><body>
<table class="stats-table">
    <thead>
        <tr><th>月</th><th>来客数</th><th>売上</th></tr>
    </thead>
    <tbody>
        <tr><td>4月</td><td>320</td><td>486000</td></tr>
        <tr><td>5月</td><td>410</td><td>612500</td></tr>
    </tbody>
</table>
</body></html>"""

# 演習の data/messy_list.html を縮小したもの。わざと「価格が無い項目」「名前が無い項目」
# 「item item-featured のように複数クラス」を混ぜてあります(実データの汚れの再現)。
MESSY_HTML = """<html><body>
<ul class="recommend-list">
    <li class="item">
        <span class="item-name">ハンドドリップ エチオピア</span>
        <span class="item-price">650円</span>
    </li>
    <li class="item">
        <span class="item-name">カフェラテ</span>
    </li>
    <li class="item item-featured">
        <span class="item-name">季節のブレンド</span>
        <span class="item-price">700円</span>
    </li>
    <li class="item">
        <span class="item-price">550円</span>
    </li>
</ul>
</body></html>"""

# それぞれをBeautifulSoupでパースしておく(unit02で学んだ html.parser)
products = BeautifulSoup(PRODUCTS_HTML, "html.parser")
table = BeautifulSoup(TABLE_HTML, "html.parser")
messy = BeautifulSoup(MESSY_HTML, "html.parser")
print("準備OK! products / table / messy の3つの soup を用意しました")

---
## 概念1: CSSセレクタ — `select` / `select_one` で「クラスの要素すべて」を一撃で

### なぜ学ぶか
実務のHTMLは「同じ構造の繰り返し」だらけです — 商品一覧、検索結果、ランキング。unit02 の `find_all("div")` だと**すべての div**が取れてしまい、そこから「class が product-card のものだけ」を自分で絞り込む手間がかかります。CSSセレクタなら「`class="product-card"` の要素すべて」を**1本の文字列**で指定できます。求人票の「大量ページからのデータ収集」で最初に効く技です。

### 解説

**CSSセレクタ**は、HTMLの要素を「どれ」と指し示すための小さな言語です(CSSで色を付ける対象を選ぶのと同じ記法)。BeautifulSoup では2つのメソッドで使います:

| メソッド | 何をする | C# の対応 |
|----------|----------|-----------|
| `soup.select("セレクタ")` | 条件に合う要素を**すべてリストで**返す(0件なら空リスト) | `xs.Where(...).ToList()` |
| `soup.select_one("セレクタ")` | 条件に合う**最初の1件**を返す(無ければ `None`) | `xs.FirstOrDefault(...)` |

セレクタ文字列の書き方(今日使う4つ):

| 書き方 | 意味 | 例 |
|--------|------|----|
| `.クラス名` | そのクラスを持つ要素 | `.product-name` |
| `タグ名` | そのタグの要素 | `h2` |
| `A B`(スペース区切り) | A の**子孫**にある B(**子孫結合子**) | `.product-grid .product-name` |
| `[属性]` / `[属性="値"]` | その属性を持つ要素 | `[data-id]` |

`.product-name`(先頭のドット)が「クラス名で選ぶ」記号です。C# のクラス名とは無関係で、HTML の `class="..."` 属性を指します。

取り出したテキストは、unit02 で学んだ `.get_text()` を使います。今回は **`.get_text(strip=True)`**(前後の空白・改行を除去して返す版)を使うのがポイント — 実HTMLはインデントの空白が入るので、`strip=True` で掃除するのが定石です。

In [ ]:
# GOAL: select で「クラスの要素すべて」を一括で取り、select_one で最初の1件を取る

# STEP 1: class="product-name" の要素をすべて取る。返り値はリスト(3件あるはず)
names = products.select(".product-name")
print("select(\".product-name\") の件数:", len(names))

# STEP 2: 各要素からテキストを取り出す。get_text(strip=True) で前後の空白を掃除
for tag in names:
    print("  -", repr(tag.get_text(strip=True)))

# STEP 3: 子孫結合子(スペース区切り)。"product-grid の中の product-price すべて"
prices = products.select(".product-grid .product-price")
print("価格:", [t.get_text(strip=True) for t in prices])

# STEP 4: select_one は最初の1件だけ(無ければ None)。属性セレクタ [data-id] も試す
first_card = products.select_one("[data-id]")
print("最初のカードの data-id:", first_card["data-id"])   # 属性の値は tag["属性名"] で取れる

### 予測してみよう

次のセルは `products.select(".product-stock")`(在庫表示すべて)を取り出し、テキストのリストにします。題材のHTML(セル2)を見返すと在庫表示は3つあります。

**実行する前に予測**: リストの中身は何になるでしょう?(在庫あり/在庫切れ のどちらが何個か)。また、存在しないクラス `.foobar` を `select` したら返り値は何になるかも予測してください。

In [ ]:
# 予測してから実行!
stocks = products.select(".product-stock")
print("在庫表示:", [t.get_text(strip=True) for t in stocks])

nothing = products.select(".foobar")   # 存在しないクラス
print("存在しないクラスの結果:", nothing, "/ 件数:", len(nothing))

`select` は該当が無くても `None` ではなく**空リスト**を返します(だから `for` で回しても安全)。ここが `select_one`(無ければ `None`)との大事な違いです。

### 書いてみる

**課題**: `products` から**商品名(`.product-name`)のテキストをすべて**集めたリストを作り、`result1` に入れてください(期待値: `["エチオピア モカ", "ブラジル サントス", "グアテマラ アンティグア"]`)。

ヒント(概念レベル): `select(".product-name")` で要素リストを取り、内包表記 `[t.get_text(strip=True) for t in ...]` でテキストに変換する。概念1の STEP 1〜2 を1行に畳むだけ。

In [ ]:
result1 = None
# ここに書く(result1 に代入する。select でtag一覧 → 内包表記でテキストのリストに)


check("概念1: CSSセレクタで一括抽出", result1,
      ["エチオピア モカ", "ブラジル サントス", "グアテマラ アンティグア"],
      hint='[t.get_text(strip=True) for t in products.select(".product-name")] の形')

---
## 概念2: 表を `list[dict]` に — thead/tbody を走査して「1行=1辞書」にする

### なぜ学ぶか
統計表・価格表・ランキングなど、`<table>` は情報の宝庫です。でも `<td>` を素朴に並べただけの2次元リスト `[["4月","320",...], ...]` は「何列目が何なのか」を人間が覚えていないと使えません。**ヘッダ(列名)をキーにした辞書のリスト** `[{"月":"4月","来客数":"320",...}, ...]` にしておくと、`row["売上"]` のように列名でアクセスでき、そのまま CSV 出力(unit05)にも流せます。スクレイピング整形の**最終ゴールの標準形**がこの `list[dict]` です。

### 解説

HTMLの表の構造(入れ子):

```
table
 ├─ thead → tr → th, th, th      ← 見出し行(列名)
 └─ tbody → tr → td, td, td      ← データ行(1 tr = 1レコード)
              tr → td, td, td
```

`th` が見出しセル、`td` がデータセルです。やることは3ステップ:

1. **ヘッダを取る**: `thead` 内の `th` を `select` してテキスト化 → `["月","来客数","売上"]`
2. **各行を取る**: `tbody` 内の各 `tr` について、その中の `td` をテキスト化 → `["4月","320","486000"]`
3. **zip で辞書化**: ヘッダと1行を `zip` で組にして `dict` にする

**`zip` と `dict()`** が新顔です。`zip(a, b)` は2つのリストを**先頭から順にペアにする**関数(C# の `a.Zip(b, ...)` と同じ)。`dict(zip(keys, values))` で「キーと値を突き合わせた辞書」が一発で作れます:

```python
dict(zip(["月","来客数"], ["4月","320"]))   # -> {"月": "4月", "来客数": "320"}
```

**要素をまたいで select する**点も重要です。`tr.select("td")` のように、`soup` 全体ではなく**特定のタグを起点に** `select` すると、その要素の**中だけ**を探します(C# で言えば部分木に対する LINQ)。

In [ ]:
# GOAL: 表を「1行=1辞書」の list[dict] に変換する3ステップを分解して見る

# STEP 1: ヘッダ(thead の中の th)をテキストのリストにする
headers = [th.get_text(strip=True) for th in table.select("thead th")]
print("ヘッダ:", headers)

# STEP 2: tbody の中の各 tr を取り、その tr の中だけの td をテキスト化(2次元リスト)
rows = []
for tr in table.select("tbody tr"):
    cells = [td.get_text(strip=True) for td in tr.select("td")]   # tr 起点で td を探す
    rows.append(cells)
print("行データ:", rows)

# STEP 3: 各行を headers と zip して辞書化 → list[dict]
records = [dict(zip(headers, row)) for row in rows]
for r in records:
    print("  ", r)

### 予測してみよう

上で作った `records`(2件の辞書のリスト)に対して、次のセルは `records[0]["来客数"]` と `records[1]["売上"]` を取り出します。

**実行する前に予測**: それぞれ何が表示されるでしょう? そして取り出した値の**型**は `int` でしょうか `str` でしょうか(HTMLのテキストは元々何型か)を考えてください。

In [ ]:
# 予測してから実行!
print("1件目の来客数:", repr(records[0]["来客数"]))
print("2件目の売上  :", repr(records[1]["売上"]))
print("来客数の型   :", type(records[0]["来客数"]).__name__)

列名でアクセスできる便利さと、値が**すべて文字列 `str`**である点に注目(`"320"` であって `320` ではない)。数値として計算するには `int(...)` での変換が要る — この「型変換」は unit05 で本格的に扱います。

### 書いてみる

**課題**: `table` から `thead th` の見出しテキストを集めたリストを作り、`result2` に入れてください(期待値: `["月", "来客数", "売上"]`)。

ヒント(概念レベル): 概念2の STEP 1 そのもの。`[th.get_text(strip=True) for th in table.select("thead th")]`。

In [ ]:
result2 = None
# ここに書く(result2 に代入する。thead th を select してテキストのリストに)


check("概念2: 表のヘッダ抽出", result2, ["月", "来客数", "売上"],
      hint='[th.get_text(strip=True) for th in table.select("thead th")] の形')

---
## 概念3: 防御的アクセス — 要素が `None` でも落ちないコード

### なぜ学ぶか
実データは**汚い**のが普通です。「価格が抜けている商品」「名前タグが無い項目」「`class="item"` が `class="item item-featured"` に揺れている項目」が平気で混ざります。`select_one` は見つからないと `None` を返すので、そこにいきなり `.get_text()` を呼ぶと **`AttributeError`(C# の `NullReferenceException` 相当)** で処理全体が止まります。1件の欠損で100件の収集が全滅 — これを防ぐのが防御的アクセスです。

### 解説

C# なら null 条件演算子 `?.` で `elem?.Text ?? "不明"` と書くところ。Python には `?.` が無いので、**明示的に `None` かどうかを見てから** `.get_text()` を呼びます:

```python
tag = item.select_one(".item-price")     # 無ければ None
price = tag.get_text(strip=True) if tag is not None else None
#       ^^^^^^^^^^^^^^^^^^^^^^^^ tag があるときだけ .get_text()、無ければ None
```

この `A if 条件 else B` は Python の**三項演算子**で、C# の `条件 ? A : B` と**順番が違う**だけ(値・条件・既定値の順)。これで `None` に `.get_text()` を呼ぶ事故を防げます。

**複数クラスの罠**: `class="item item-featured"` の要素は「item と item-featured の**両方の**クラスを持つ」という意味です。CSSセレクタ `.item` は「**item を含む**要素」を選ぶので、`.item` はこの複数クラス要素も**ちゃんと拾います**(C# の `HashSet<string>` に "item" が含まれるか、のイメージ)。だから `select(".item")` で全項目が漏れなく取れます。

まとめると、繰り返し構造を集める定石は「`select(".item")` で全件 → 各件で `select_one` + `None` ガードして辞書に詰める」です。

In [ ]:
# GOAL: 欠損のある messy リストを、None ガードしながら list[dict] に整形する

# STEP 1: .item を持つ li をすべて取る。item item-featured(複数クラス)も拾えるか確認
items = messy.select(".item")
print(".item の件数:", len(items), "(複数クラスの項目も含めて全部で4件のはず)")

# STEP 2: 1件を None ガード付きで辞書化する関数(欠損しても落ちない)
def to_record(item):
    name_tag = item.select_one(".item-name")
    price_tag = item.select_one(".item-price")
    # tag が None のときは .get_text() を呼ばずに既定値にフォールバック
    name = name_tag.get_text(strip=True) if name_tag is not None else "(名称不明)"
    price = price_tag.get_text(strip=True) if price_tag is not None else None
    return {"name": name, "price": price}

# STEP 3: 全件に適用。名前欠損・価格欠損が混ざっていても最後まで走り切る
for item in items:
    print("  ", to_record(item))

### 予測してみよう

次のセルは、**名前タグが無い項目**(4件目、価格だけの `<li>`)に対して `select_one(".item-name")` を呼び、その結果と、`None` ガード**無し**で `.get_text()` を呼んだ場合に何が起きるかを `try/except` で見せます。

**実行する前に予測**: `select_one(".item-name")` は何を返すでしょう? そしてガード無しで `.get_text()` を呼ぶと、どんな種類のエラーになるでしょう?

In [ ]:
# 予測してから実行!
items = messy.select(".item")
no_name_item = items[3]   # 4件目 = 名前タグが無く価格だけの項目

tag = no_name_item.select_one(".item-name")
print("select_one(\".item-name\") の結果:", tag)   # None のはず

try:
    tag.get_text(strip=True)   # None に .get_text() を呼ぶと…
except AttributeError as e:
    print("ガード無しで呼ぶと例外:", type(e).__name__, "->", e)

`None.get_text()` が `AttributeError` で落ちる — これが「防御的アクセス」が要る理由です。`if tag is not None else ...` の一言で、この事故を全件で防げます。

### 書いてみる

**課題**: `messy` の**名前も価格も両方揃っている項目だけ**の件数を数えて `result3` に入れてください(期待値: `2`。1件目「ハンドドリップ エチオピア」と3件目「季節のブレンド」だけが両方揃っている)。

ヒント(概念レベル): `messy.select(".item")` で全件を回し、各件で `select_one(".item-name")` と `select_one(".item-price")` が**両方とも `None` でない**ものだけカウントする。`if A is not None and B is not None:` で1ずつ足す。

In [ ]:
result3 = None
# ここに書く(result3 に代入する。name と price が両方そろった .item の件数)


check("概念3: 防御的アクセスで件数集計", result3, 2,
      hint='for item in messy.select(".item"): name=item.select_one(".item-name"); price=item.select_one(".item-price"); 両方 None でないとき件数+1')

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| CSSセレクタ | `select(".クラス")` で該当要素を**すべて**、`select_one` で最初の1件。子孫は空白、属性は `[..]` | `Where(...).ToList()` / `FirstOrDefault()` |
| 表→list[dict] | `thead th` でヘッダ、`tbody tr`→`td` で行、`dict(zip(headers,row))` で辞書化 | `DataTable` を列名キーの `Dictionary` に |
| 防御的アクセス | `select_one` は無いと `None`。`x.get_text() if x else 既定値` で事故防止 | null条件演算子 `?.` / `?? 既定値` |

**この先どこで使うか**:
- **今日の演習(ex01〜ex04)**は、まさにこの3つ(CSSセレクタ / 表→list[dict] / 防御的アクセス)を `products.html` / `table_stats.html` / `messy_list.html` の実ファイルに対してやります。このレッスンの題材はその縮小版なので、**同じ構造**に演習で再会します。
- **unit05(堅牢化とCSV)**では、今日の防御的アクセスをさらに一般化した**例外処理(try/except)・欠損のスキップとログ**へ発展させ、`list[dict]` を `csv.DictWriter` でCSVに書き出します。今日「値がすべて `str`」だと気づいた点も、unit05 の型変換(`"1200円"`→`1200`)につながります。
- **unit06(キャップストーン)**では、複数ページをまたいで今日の「繰り返し構造→list[dict]」を何度も適用し、1本のCSVカタログにまとめます。

**次**: 演習 `ex01_css_select.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/web-scraping/unit03-selectors-and-tables/tests/test_ex01.py -q`